In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

WINDOW = 24
BATCH_SIZE = 512
EPOCHS = 30
N_SPLITS = 5
MAX_SAMPLES = 400000

data = pd.read_csv("Data/Final_Data.csv")
if "SiteKey" in data.columns:
    data = data.drop(columns=["SiteKey"])

Y_NOW = data["Residual_now"]
Y_TRUE_NOW = data["CapacityFactor"]
Y_MULTI = data[[
    "Residual_15min",
    "Residual_1h",
    "Residual_3h"
]]
Y_TRUE_MULTI = data[[
    "Target_15min",
    "Target_1h",
    "Target_3h"
]]
drop_cols = [
    "CapacityFactor",
    "Target_15min",
    "Target_1h",
    "Target_3h",
    "Residual_15min",
    "Residual_1h",
    "Residual_3h",
    "Residual_now",
    "SolarGeneration"
]
features = data.drop(columns=drop_cols)
features = features.iloc[-MAX_SAMPLES:]

Y_NOW = Y_NOW.iloc[-MAX_SAMPLES:]
Y_TRUE_NOW = Y_TRUE_NOW.iloc[-MAX_SAMPLES:]
Y_MULTI = Y_MULTI.iloc[-MAX_SAMPLES:]
Y_TRUE_MULTI = Y_TRUE_MULTI.iloc[-MAX_SAMPLES:]

lag_now = features["Lag_1"].values.astype(np.float32)
lag_15 = features["Lag_1"].values.astype(np.float32)
lag_1h = features["Lag_2"].values.astype(np.float32)
lag_3h = features["Lag_3"].values.astype(np.float32)

features = features.astype(np.float32)
X_values = features.values.astype(np.float32)
Y_NOW_values = Y_NOW.values.astype(np.float32)
Y_TRUE_NOW_values = Y_TRUE_NOW.values.astype(np.float32)
Y_MULTI_values = Y_MULTI.values.astype(np.float32)
Y_TRUE_MULTI_values = Y_TRUE_MULTI.values.astype(np.float32)

scaler = StandardScaler()
X_values = scaler.fit_transform(X_values)
X_values = X_values.astype(np.float32)

def create_dataset(X, y, start_idx, end_idx):
    dataset = tf.keras.utils.timeseries_dataset_from_array(
        data=X,
        targets=y[WINDOW:end_idx],
        sequence_length=WINDOW,
        sequence_stride=1,
        sampling_rate=1,
        batch_size=BATCH_SIZE,
        start_index=start_idx,
        end_index=end_idx - 1,
        shuffle=False
    )
    return dataset

def build_single_lstm(num_features):
    model = models.Sequential([
        layers.Input(shape=(WINDOW, num_features)),
        layers.LSTM(
            128,
            return_sequences=True
        ),
        layers.Dropout(0.2),
        layers.LSTM(64),
        layers.Dropout(0.2),
        layers.Dense(
            64,
            activation="relu"
        ),
        layers.Dense(1)
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="mse"
    )
    return model

def build_multi_lstm(num_features):
    model = models.Sequential([
        layers.Input(shape=(WINDOW, num_features)),
        layers.LSTM(
            128,
            return_sequences=True
        ),
        layers.Dropout(0.2),
        layers.LSTM(64),
        layers.Dropout(0.2),
        layers.Dense(
            64,
            activation="relu"
        ),
        layers.Dense(3)
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss=tf.keras.losses.Huber()
    )
    return model

split_size = len(X_values) // (N_SPLITS + 1)

mae_model_now = 0
mae_naive_now = 0
r2_now = 0

mae_model_multi = np.zeros(3)
mae_naive_multi = np.zeros(3)
r2_multi = np.zeros(3)
horizons = [
    "15 min",
    "1 hour",
    "3 hour"
]

for i in range(N_SPLITS):

    print(f"\n==============================")
    print(f"========== SPLIT {i+1} ==========")
    print(f"==============================")

    train_end = split_size * (i + 1)
    test_end = split_size * (i + 2)

    train_dataset_now = create_dataset(
        X_values,
        Y_NOW_values,
        0,
        train_end
    )

    test_dataset_now = create_dataset(
        X_values,
        Y_NOW_values,
        train_end,
        test_end
    )

    train_dataset_multi = create_dataset(
        X_values,
        Y_MULTI_values,
        0,
        train_end
    )

    test_dataset_multi = create_dataset(
        X_values,
        Y_MULTI_values,
        train_end,
        test_end
    )

    early_stop = callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    reduce_lr = callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        patience=3,
        factor=0.5,
        verbose=1
    )

    print("\nTRAINING NOW MODEL")

    model_now = build_single_lstm(
        num_features=X_values.shape[1]
    )

    model_now.fit(
        train_dataset_now,
        validation_data=test_dataset_now,
        epochs=EPOCHS,
        callbacks=[
            early_stop,
            reduce_lr
        ],
        verbose=1
    )

    print("\nTRAINING MULTI-HORIZON MODEL")

    model_multi = build_multi_lstm(
        num_features=X_values.shape[1]
    )

    model_multi.fit(
        train_dataset_multi,
        validation_data=test_dataset_multi,
        epochs=EPOCHS,
        callbacks=[
            early_stop,
            reduce_lr
        ],
        verbose=1
    )

    residual_pred_now = model_now.predict(
        test_dataset_now,
        verbose=0
    ).flatten()

    residual_pred_multi = model_multi.predict(
        test_dataset_multi,
        verbose=0
    )


    test_start_real = train_end + WINDOW
    test_end_real = test_end

    true_now = Y_TRUE_NOW_values[
        test_start_real:test_end_real
    ]

    true_multi = Y_TRUE_MULTI_values[
        test_start_real:test_end_real
    ]

    lag_test_now = lag_now[
        test_start_real:test_end_real
    ]

    lag_test_15 = lag_15[
        test_start_real:test_end_real
    ]

    lag_test_1h = lag_1h[
        test_start_real:test_end_real
    ]

    lag_test_3h = lag_3h[
        test_start_real:test_end_real
    ]

    pred_now = lag_test_now + residual_pred_now
    pred_15 = lag_test_15 + residual_pred_multi[:, 0]
    pred_1h = lag_test_1h + residual_pred_multi[:, 1]
    pred_3h = lag_test_3h + residual_pred_multi[:, 2]

    preds_multi = [
        pred_15,
        pred_1h,
        pred_3h
    ]

    lags_multi = [
        lag_test_15,
        lag_test_1h,
        lag_test_3h
    ]

    fold_mae_model_now = mean_absolute_error(
        true_now,
        pred_now
    )

    fold_mae_naive_now = mean_absolute_error(
        true_now,
        lag_test_now
    )

    fold_r2_now = r2_score(
        true_now,
        pred_now
    )

    mae_model_now += fold_mae_model_now
    mae_naive_now += fold_mae_naive_now
    r2_now += fold_r2_now

    for h in range(3):

        fold_mae_model = mean_absolute_error(
            true_multi[:, h],
            preds_multi[h]
        )

        fold_mae_naive = mean_absolute_error(
            true_multi[:, h],
            lags_multi[h]
        )

        fold_r2 = r2_score(
            true_multi[:, h],
            preds_multi[h]
        )

        mae_model_multi[h] += fold_mae_model
        mae_naive_multi[h] += fold_mae_naive
        r2_multi[h] += fold_r2

    print("\n==============================")
    print("NOW PREDICTIONS")
    print("==============================")

    print("Pred :", pred_now[:5])
    print("True :", true_now[:5])
    print("Lag  :", lag_test_now[:5])

    print("\n==============================")
    print("MULTI-HORIZON PREDICTIONS")
    print("==============================")

    print("\n15 MIN")
    print(pred_15[:5])

    print("\n1 HOUR")
    print(pred_1h[:5])

    print("\n3 HOUR")
    print(pred_3h[:5])

mae_model_now /= N_SPLITS
mae_naive_now /= N_SPLITS
r2_now /= N_SPLITS

mae_model_multi /= N_SPLITS
mae_naive_multi /= N_SPLITS
r2_multi /= N_SPLITS

print("\n===================================")
print("FINAL NOW LSTM RESULTS")
print("===================================")

print("Naive MAE:", mae_naive_now)
print("Model MAE:", mae_model_now)
print("R2:", r2_now)

print("\n===================================")
print("FINAL MULTI-HORIZON LSTM RESULTS")
print("===================================")

for i, h in enumerate(horizons):

    print(f"\nLSTM Horizon: {h}")

    print("Naive MAE:", mae_naive_multi[i])

    print("Model MAE:", mae_model_multi[i])

    print("R2:", r2_multi[i])


========== SPLIT 1 ==========

TRAINING NOW MODEL
Epoch 1/30
131/131 ━━━━━━━━━━━━━━━━━━━━ 46s 303ms/step - loss: 0.0021 - val_loss: 4.0980e-04 - learning_rate: 0.0010
Epoch 2/30
131/131 ━━━━━━━━━━━━━━━━━━━━ 39s 297ms/step - loss: 4.2150e-04 - val_loss: 2.7312e-04 - learning_rate: 0.0010
Epoch 3/30
131/131 ━━━━━━━━━━━━━━━━━━━━ 46s 353ms/step - loss: 2.9593e-04 - val_loss: 2.5303e-04 - learning_rate: 0.0010
Epoch 4/30
131/131 ━━━━━━━━━━━━━━━━━━━━ 34s 263ms/step - loss: 2.5088e-04 - val_loss: 2.3386e-04 - learning_rate: 0.0010
Epoch 5/30
130/131 ━━━━━━━━━━━━━━━━━━━━ 0s 188ms/step - loss: 2.3712e-04
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
131/131 ━━━━━━━━━━━━━━━━━━━━ 36s 273ms/step - loss: 2.2645e-04 - val_loss: 2.2872e-04 - learning_rate: 0.0010
Epoch 6/30
131/131 ━━━━━━━━━━━━━━━━━━━━ 35s 267ms/step - loss: 2.1158e-04 - val_loss: 2.3112e-04 - learning_rate: 5.0000e-04
Epoch 7/30
131/131 ━━━━━━━━━━━━━━━━━━━━ 36s 275ms/step - loss: 2.0621e-04 - val_loss: